# Playfit Intelligence Lab — 05: MLflow Tracking, A/B Testing & LLM Explainability

Este notebook demuestra tres capacidades avanzadas de MLOps y evaluación:

1. **MLflow Experiment Tracking**: Logging de parámetros, métricas y artefactos
2. **A/B Test Simulator**: Confrontación estadística entre variantes del modelo
3. **LLM Explainability**: Explicaciones narrativas con GPT-4o-mini

In [ ]:
import sys; sys.path.insert(0, '..')
import warnings; warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import polars as pl

from src.features.game_features import build_feature_matrix, compute_popularity_score, compute_richness_score
from src.models.content_based import build_content_model
from src.models.hybrid import HybridRecommender

print("Librerías cargadas.")
print(f"MLflow version: {__import__('mlflow').__version__}")

## 1. MLflow Experiment Tracking

Creamos un experimento y corremos grid search sobre α, β, γ.

In [ ]:
import mlflow

mlflow.set_experiment("playfit-hybrid-demo")
print(f"MLflow tracking URI: {mlflow.get_tracking_uri()}")
print("To view: run 'mlflow ui' in terminal, open http://localhost:5000")

In [ ]:
from src.training.experiment_grid import run_grid_search

# This runs 10 experiments and logs params/metrics/figures to MLflow
df_results = run_grid_search(experiment_name="playfit-hybrid-demo")
print(f"\nBest config: α={df_results.iloc[0]['alpha']}, β={df_results.iloc[0]['beta']}, γ={df_results.iloc[0]['gamma']}")

## 2. A/B Test Simulator

Comparamos dos variantes del modelo con significancia estadística.

In [ ]:
from src.evaluation.ab_test import ABTestSimulator

fm = compute_richness_score(compute_popularity_score(build_feature_matrix()))
cm = build_content_model(fm, n_components=100)

rec_a = HybridRecommender(alpha=0.5, beta=0.4, gamma=0.1)
rec_a.fit(fm, cm)

rec_b = HybridRecommender(alpha=0.7, beta=0.2, gamma=0.1)
rec_b.fit(fm, cm)

ab = ABTestSimulator(rec_a, rec_b, n_users=100)
result = ab.simulate(k=10)

print("=== A/B Test: Variant A (α=0.5) vs Variant B (α=0.7) ===")
print(f"Users: {result['n_users']}")
print(f"Model A mean NDCG@10: {result['model_a']['mean_ndcg']:.4f}")
print(f"Model B mean NDCG@10: {result['model_b']['mean_ndcg']:.4f}")
print(f"Lift: {result['lift_pct']:.2f}% ({result['lift']:.4f})")
print(f"95% CI: [{result['ci_95'][0]:.4f}, {result['ci_95'][1]:.4f}]")
print(f"p-value: {result['p_value']:.4f}")
print(f"Statistically significant: {result['significant']}")
print(f"Wins A: {result['n_wins_a']}, Wins B: {result['n_wins_b']}, Ties: {result['n_ties']}")

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

means = [result['model_a']['mean_ndcg'], result['model_b']['mean_ndcg']]
stds = [result['model_a']['std_ndcg'], result['model_b']['std_ndcg']]
bars = ax1.bar(['Variant A (α=0.5)', 'Variant B (α=0.7)'], means, yerr=stds,
               capsize=10, color=['#3498db', '#e74c3c'])
ax1.set_ylabel('Mean NDCG@10')
ax1.set_title('A/B Test: Mean Performance')
for bar, mean, std in zip(bars, means, stds):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + std + 0.01,
             f'{mean:.4f}', ha='center', fontsize=11)

wins = [result['n_wins_a'], result['n_wins_b'], result['n_ties']]
ax2.bar(['Variant A Wins', 'Variant B Wins', 'Ties'], wins,
        color=['#3498db', '#e74c3c', '#95a5a6'])
ax2.set_ylabel('Users')
ax2.set_title(f'Win Distribution (p={result["p_value"]:.3f})')

plt.tight_layout()
plt.savefig('../reports/figures/ab_test_results.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. LLM Explainability

Generamos explicaciones narrativas usando un LLM y comparamos con el baseline rule-based.

In [ ]:
from src.evaluation.explainability import make_explanation, get_game_details
from src.evaluation.llm_explainability import LLMExplainer, compare_explanations

fm = compute_richness_score(compute_popularity_score(build_feature_matrix()))
cm = build_content_model(fm, n_components=100)
rec = HybridRecommender(alpha=0.5, beta=0.4, gamma=0.1)
rec.fit(fm, cm)

results = rec.recommend(['the_legend_of_zelda_breath_of_the_wild'], k=5)

explainer = LLMExplainer(model="gpt-4o-mini")
print("Generating LLM explanations... (requires OPENROUTER_API_KEY or OPENAI_API_KEY)")

for r in results:
    details = get_game_details(r['game_id'])
    rule_expl = make_explanation(r)
    print(f"\n--- {details.get('title', r['game_id'])} ---")
    print(f"Rule-based: {rule_expl}")
    llm_expl = explainer.explain(r)
    print(f"LLM: {llm_expl}")
    comp = compare_explanations(rule_expl, llm_expl)
    print(f"  (LLM longer: {comp['llm_is_longer']}, chars: {comp['rule_based_chars']} vs {comp['llm_based_chars']})")

## 4. Resumen de Capacidades Añadidas

| Capacidad | Herramienta | Estado |
|-----------|------------|--------|
| Experiment tracking | MLflow 3.14 | ✅ Logs a `mlruns/` |
| Grid search | 10 configuraciones α/β/γ | ✅ Auto-logueado |
| A/B Testing | Bootstrap + CI + p-value | ✅ |
| LLM Explainability | GPT-4o-mini via OpenRouter | ✅ (requiere API key) |
| MLflow UI | `mlflow ui` | ✅ http://localhost:5000 |